In [3]:
# Imports
import os, pickle, torch
import matplotlib.pyplot as plt
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, TextIteratorStreamer
import matplotlib.colors as pltc
import numpy as np
import torch.optim as optim
from torch.utils.tensorboard import SummaryWriter
import datetime
import time
LOCATION = '/igli/raw_logits_v2/'
device = 'cuda:0'

In [455]:
class quantizer_4b_torch:
    
    def quantize(self,a): # it is assumed that the distribution of the values is normal
        is_negative=torch.sign(a)<0
        
        x=torch.sqrt(torch.abs(a))*5.3-2.1
        x=torch.round(x)
        x=torch.clip(x,0,7).to(torch.int8)
        
        x=x+8*is_negative.to(torch.int8)
        return x
    
    def decode(self,x):
        is_negative=x>=8
        a=(( torch.remainder(x,8).to(torch.float16)+2.1)/5.3)**2
        a=a*(1-is_negative.to(torch.int8)*2)
        return a
    
    def quantize_compress(self,a):
        ind=self.quantize(a)
        comp=ind[:,0::2]*16+ind[:,1::2]
        return comp.to(torch.uint8)

    def load_decode(self,data):
        ind=torch.zeros((data.shape[0],data.shape[1]*2),dtype=torch.uint8,device=data.device)
        ind[:,0::2]=torch.floor_divide(data,16)
        ind[:,1::2]=torch.remainder(data,16)
        return self.decode(ind)

class quantizer_2b_torch:
    
    def quantize(self,a): # it is assumed that the distribution of the values is normal
        is_negative=torch.sign(a)<0
        
        x=torch.sqrt(torch.abs(a))*1.63-1
        x=torch.round(x)
        x=torch.clip(x,0,1).to(torch.int8)
        
        x=x+2*is_negative.to(torch.int8)
        return x
    
    def decode(self,x):
        is_negative=x>=2
        a=(( torch.remainder(x,2).to(torch.float16)+1)/1.63)**2
        a=a*(1-is_negative.to(torch.int8)*2)
        return a
    
    def quantize_compress(self,a):
        ind=self.quantize(a)
        comp=ind[:,0::4]*64+ind[:,1::4]*16+ind[:,2::4]*4+ind[:,3::4]*1
        return comp.to(torch.uint8)

    def load_decode(self,data):
        ind=torch.zeros((data.shape[0],data.shape[1]*4),dtype=torch.uint8,device=data.device)
        ind[:,0::4]=torch.remainder(torch.floor_divide(data,64),4)
        ind[:,1::4]=torch.remainder(torch.floor_divide(data,16),4)
        ind[:,2::4]=torch.remainder(torch.floor_divide(data, 4),4)
        ind[:,3::4]=torch.remainder(torch.floor_divide(data, 1),4)
        return self.decode(ind)

class quantizer_1b_torch:
    
    def quantize(self,a):
        is_negative=torch.sign(a)<0
        return (is_negative).to(torch.uint8)
    
    def decode(self,x):
        is_negative=x>=1
        a=torch.remainder(x,1).to(torch.float16)+0.8
        a=a*(1-is_negative.to(torch.int8)*2)
        return a
    
    def quantize_compress(self,a):
        ind=self.quantize(a)
        comp=ind[:,0::8]*128+ind[:,1::8]* 64+ind[:,2::8]* 32+ind[:,3::8]* 16+ind[:,4::8]*  8+ind[:,5::8]*  4+ind[:,6::8]*  2+ind[:,7::8]*  1
        return comp.to(torch.uint8)

    def load_decode(self,data):
        ind=torch.zeros((data.shape[0],data.shape[1]*8),dtype=torch.uint8,device=data.device)
        ind[:,0::8]=torch.remainder(torch.floor_divide(data,128),2)
        ind[:,1::8]=torch.remainder(torch.floor_divide(data, 64),2)
        ind[:,2::8]=torch.remainder(torch.floor_divide(data, 32),2)
        ind[:,3::8]=torch.remainder(torch.floor_divide(data, 16),2)
        ind[:,4::8]=torch.remainder(torch.floor_divide(data,  8),2)
        ind[:,5::8]=torch.remainder(torch.floor_divide(data,  4),2)
        ind[:,6::8]=torch.remainder(torch.floor_divide(data,  2),2)
        ind[:,7::8]=torch.remainder(torch.floor_divide(data,  1),2)
        return self.decode(ind)
    
class quantizer_mixed_torch:
    
    def __init__(self,vector_length):
        self.proportion_4b=1/8
        self.proportion_2b=4/8
        self.proportion_1b=3/8
        self.count_4b=int(round(self.proportion_4b*vector_length))
        self.count_2b=int(round(self.proportion_2b*vector_length))
        self.count_1b=int(round(self.proportion_1b*vector_length))
        self.q4=quantizer_4b_torch()
        self.q2=quantizer_2b_torch()
        self.q1=quantizer_1b_torch()
                
    def quantize(self,a):
        a4=self.q4.quantize(a[:,              : self.count_4b])
        a2=self.q2.quantize(a[:, self.count_4b:-self.count_1b])
        a1=self.q1.quantize(a[:,-self.count_1b:              ])      
        return torch.concatenate([a4,a2,a1],axis=1)
    
    def decode(self,index):
        index4=self.q4.decode(index[:,              : self.count_4b])
        index2=self.q2.decode(index[:, self.count_4b:-self.count_1b])
        index1=self.q1.decode(index[:,-self.count_1b:              ])      
        return torch.concatenate([index4,index2,index1],axis=1)

    def quantize_compress(self,a):
        comp4=self.q4.quantize_compress(a[:,              : self.count_4b])
        comp2=self.q2.quantize_compress(a[:, self.count_4b:-self.count_1b])
        comp1=self.q1.quantize_compress(a[:,-self.count_1b:              ])      
        return torch.concatenate([comp4,comp2,comp1],axis=1)

    def load_decode(self,comp):
        comp4=comp[:, 0                 : self.count_4b*4//8]
        comp2=comp[:, self.count_4b*4//8:-self.count_1b*1//8]
        comp1=comp[:,-self.count_1b*1//8:                   ]
        a4=self.q4.load_decode(comp4)
        a2=self.q2.load_decode(comp2)
        a1=self.q1.load_decode(comp1)
        return torch.concatenate([a4,a2,a1],axis=1)

class representation_whitening_torch:
    
    def fit(self,a):
        self.mn=a.mean(dim=0)
        
        # low memory covariance calculation
        cov=torch.eye(a.shape[1],dtype=torch.float32,device=a.device)*0
        for i in tqdm(range(0,a.shape[0],10000)):
            aslice=(a[i:i+10000,:].to(torch.float32)-self.mn)
            cov+=( aslice.T@aslice)/a.shape[0]
        
        
        s2,self.q=torch.linalg.eigh(cov)
        self.q=self.q.to(torch.float16)
        self.s=torch.sqrt(s2).to(torch.float16)
        
        if self.s[0]<self.s[-1]:
            self.s=torch.flip(self.s,dims=[0])
            self.q=torch.flip(self.q,dims=[1])
       
    def encode(self,a):
        return (a-self.mn)@self.q /self.s
    
    def decode(self,e):
        return (self.s*e)@self.q.T+self.mn
    
    def rescale(self,e):
        return self.s*e
    
    def save_state_dict(self,path):
        torch.save({'mn':self.mn,'q':self.q,'s':self.s}, path)

    def load_state_dict(self,path):
        data=torch.load(path)
        self.mn=data['mn']
        self.q =data['q' ]
        self.s =data['s' ]
        
    def to(self, d):
        self.mn = self.mn.to(d)
        self.q = self.q.to(d)
        self.s = self.s.to(d)

In [4]:
LOCATION = "/igli/raw_logits_v2"
t = !find {LOCATION} -name "*_embeddings_new"
tt = [os.path.basename(x)[:4] for x in t]

In [15]:
fff = np.array([int(x) for x in tt])


[182 186 190 194 198 202 206 210 214 218 222 226 230 234 238 242 246 250
 254 258 262 266 270 274 278 282 286 290 294 298 302 306 310 314 318 322
 326 330 334 338 342 346 350 354 358 362 366 370 374 378 382 386 390 394
 398 402 406 410 414 418 422 426 430 434 438 442 446 450 454 458 462 466
 470 474 478 482 486 490 494 498 502 506 510 514 518 520 522 524 526 529
 530 534 535 538 540 542 544 545 546 549 550 551 553 554 555 556 557 558
 559 560 561 562 563 564 565 566 567 568 569 570 571 572 573 574 575 576
 577 578 579 580 581 582 583 584 585 586 587 588 589 590 591 592 593 594
 595 596 597 598 599 600 601 602 603 604 605 606 607 608 609 610 611 612
 613 614 615 616 617 618 619 620 621 622 623 624 625 626 627 628 629 630
 631 632 633 634 635 636 637 638 639 640 641 642 643 644 645 646 647 648
 649 650 651 652 653 654 655 656 657 658 659 660 661 662 663 664 665 666
 667 668 669 670 671 672 673 674 675 676 677 678 679 680 681 682 683 684
 685 686 687 688 689 690 691 692 693 694 695 696 69

In [16]:
locs = (fff > 450).nonzero()[0]

In [20]:
locs.shape

(2007,)

In [22]:
gaps = locs[1:] - locs[:-1]
gaps = (gaps == 1)

In [37]:
locs[101:]

array([ 551,  553,  554, ..., 2455, 2456, 2457])

In [35]:
gaps[101:].all()

False

In [4]:
tokens_of_interest = np.fromfile('tokens_of_interest', dtype=np.int32)

In [456]:
q = quantizer_4b_torch()
w = representation_whitening_torch()
w.load_state_dict('wtorch.pth')

In [162]:
tt

['0000',
 '0001',
 '0004',
 '0002',
 '0005',
 '0003',
 '0008',
 '0006',
 '0009',
 '0007',
 '0012',
 '0010',
 '0013',
 '0011',
 '0373',
 '0340',
 '0016',
 '0014',
 '0017',
 '0015',
 '0354',
 '0359',
 '0377',
 '0020',
 '0018',
 '0021',
 '0019',
 '0024',
 '0022',
 '0025',
 '0023',
 '0026',
 '0028',
 '0029',
 '0027',
 '0030',
 '0032',
 '0033',
 '0031',
 '0034',
 '0037',
 '0036',
 '0035',
 '0038',
 '0041',
 '0040',
 '0039',
 '0042',
 '0045',
 '0043',
 '0044',
 '0046',
 '0049',
 '0047',
 '0048',
 '0050',
 '0053',
 '0051',
 '0052',
 '0054',
 '0057',
 '0055',
 '0058',
 '0056',
 '0061',
 '0059',
 '0062',
 '0060',
 '0065',
 '0063',
 '0066',
 '0069',
 '0064',
 '0067',
 '0070',
 '0073',
 '0068',
 '0071',
 '0074',
 '0077',
 '0072',
 '0075',
 '0078',
 '0081',
 '0076',
 '0079',
 '0082',
 '0085',
 '0080',
 '0083',
 '0086',
 '0089',
 '0087',
 '0084',
 '0090',
 '0093',
 '0091',
 '0088',
 '0094',
 '0097',
 '0095',
 '0098',
 '0092',
 '0101',
 '0099',
 '0102',
 '0096',
 '0105',
 '0103',
 '0106',
 '0109',
 

In [7]:
# Inverted Index Generation
nfiles=len(tt)

for file_id_start in tqdm(range(0, nfiles, 50)):
    filenames =  tt[file_id_start:file_id_start+50]
    tokens=np.zeros( (1000,1024,11,len(filenames)),dtype=np.int32)
    for j, jj in tqdm(enumerate(filenames)):
        ts=np.fromfile(f"{LOCATION}/{jj}_id_t" , dtype=np.int32).reshape(-1, 1024, 100)[..., :10]
        gs=np.fromfile(f"{LOCATION}/{jj}_id_gt", dtype=np.int32).reshape(1000, 1024)
        tokens[:,:,0:10,j]=ts
        tokens[:,:,-  1,j]=gs

    file_id   =np.tile(np.array([int(x)for x in filenames],dtype=np.uint16).reshape( (1   ,   1,1,50) ), (1000,1024,11,  1) )
    example_id=np.tile(np.arange(1000,                     dtype=np.uint16).reshape( (1000,   1,1, 1) ), (   1,1024,11, 50) )
    position  =np.tile(np.arange(1024,                     dtype=np.uint16).reshape( (1   ,1024,1, 1) ), (1000,   1,11, 50) )
    
    tokens_cuda = torch.tensor(tokens).to('cuda:3').reshape(-1)
    toi_cuda = torch.tensor(tokens_of_interest).to('cuda:3')
    IX=torch.argsort(tokens_cuda).cpu().numpy()
    file_id    = file_id   .reshape(-1)[IX]
    example_id = example_id.reshape(-1)[IX]
    position   = position  .reshape(-1)[IX]
    tokens     = tokens    .reshape(-1)[IX]
    mask = np.isin(tokens, tokens_of_interest)
    file_id    = file_id[mask].reshape(-1,1)
    example_id = example_id[mask].reshape(-1,1)
    position   = position[mask].reshape(-1,1)
    indices = np.concatenate((file_id, example_id, position), axis=-1)
    tokens     = tokens[mask]
    cutoffs = np.where(tokens[:-1] != tokens[1:])[0] + 1
    start = 0
    for end in tqdm(cutoffs):
        token_to_save = tokens[start]
        with open(f"indexfiles/{token_to_save}_{file_id_start//50}_index", "w") as myfile:
            indices[start:end].tofile(myfile)
        start = end

  0%|                                                    | 0/50 [00:00<?, ?it/s]
0it [00:00, ?it/s]
1it [00:01,  1.19s/it]
2it [00:01,  1.37it/s]
3it [00:02,  1.70it/s]
4it [00:02,  1.94it/s]
5it [00:02,  2.10it/s]
6it [00:03,  2.21it/s]
7it [00:03,  2.29it/s]
8it [00:04,  2.34it/s]
9it [00:04,  2.38it/s]
10it [00:04,  2.41it/s]
11it [00:05,  2.47it/s]
12it [00:05,  2.50it/s]
13it [00:06,  2.53it/s]
14it [00:06,  2.55it/s]
15it [00:06,  2.56it/s]
16it [00:07,  2.57it/s]
17it [00:07,  2.58it/s]
18it [00:07,  2.59it/s]
19it [00:08,  2.60it/s]
20it [00:08,  2.61it/s]
21it [00:09,  2.61it/s]
22it [00:09,  2.62it/s]
23it [00:09,  2.62it/s]
24it [00:10,  2.62it/s]
25it [00:10,  2.62it/s]
26it [00:10,  2.62it/s]
27it [00:11,  2.62it/s]
28it [00:11,  2.62it/s]
29it [00:12,  2.62it/s]
30it [00:12,  2.62it/s]
31it [00:12,  2.62it/s]
32it [00:13,  2.62it/s]
33it [00:13,  2.62it/s]
34it [00:14,  2.63it/s]
35it [00:14,  2.63it/s]
36it [00:14,  2.63it/s]
37it [00:15,  2.62it/s]
38it [00:15,  2.63it/

ValueError: cannot reshape array of size 8 into shape (1,1,1,50)

In [142]:
# Post Processing
path = "indexfiles"
for filename in os.listdir(path):
    index_file = np.fromfile(f"{path}/{filename}", dtype=np.uint16).reshape(-1,3)
    index_file1 = torch.unique(torch.tensor(index_file.astype('int32'), device=device), dim=0).cpu().numpy().astype('uint16')
    index_file1.tofile(f"{path}/{filename}")

In [370]:
def lexsort(tensor):
    # Lexicographical sort: Sort by the last column first, then by the previous columns
    # Reverse sorting order from the last to the first column
    sorted_indices = torch.argsort(tensor[:, -1])  # Start with the last column
    
    for i in range(tensor.shape[1] - 2, -1, -1):
        sorted_indices = sorted_indices[torch.argsort(tensor[sorted_indices, i])]
    
    # Apply the sorted indices to the original tensor
    sorted_tensor = tensor[sorted_indices]
    
    return sorted_tensor, sorted_indices


In [531]:
np.arange(5).cumsum()

array([ 0,  1,  3,  6, 10])

In [538]:
# Post Processing to retrieve the index in the student probabilities file
path = "indexfiles"
# for file_id_start in tqdm(range(0, len(tt), 50)):
inverted_fname_index = np.zeros(3000, dtype=np.uint32)
for file_id_start in tqdm(range(0, 1, 50)):
    i = file_id_start // 50
    filenames = tt[file_id_start : file_id_start+50]
    
    inverted_fname_index[[int(x) for x in filenames]] = np.arange(len(filenames), dtype=np.uint32)
    
    indexes_ith_chunk = [np.fromfile(f"{LOCATION}/{x}_student_index_new", dtype=np.uint16).reshape(-1,2) for x in filenames]
    indexes_ith_chunk_offsets = np.array([0] + [x.shape[0] for x in indexes_ith_chunk]).cumsum(dtype=np.uint32)
    
    file_index = [np.repeat([int(x)], y.shape[0]).reshape(-1,1) for x,y in zip(filenames, indexes_ith_chunk)]
    indexes_ith_chunk_final = [np.concatenate((x,y), axis=-1) for x,y in zip(file_index, indexes_ith_chunk)]
    indexes_ith_chunk_final = np.concatenate(indexes_ith_chunk_final)
    full = torch.tensor(indexes_ith_chunk_final, device=device).int()
    for token_id in tqdm(tokens_of_interest[:1]):
        index_token = np.fromfile(f"{INDEXES_LOCATION}/{token_id}_{i}_index", dtype=np.uint16).reshape(-1,3)
        partial = torch.tensor(index_token, device=device).int()
        
        mask = torch.isin((full * torch.tensor([10000**2, 10000, 1], device=device)).sum(dim=-1), (partial * torch.tensor([10000**2, 10000, 1], device=device)).sum(-1), assume_unique=True)
        original_indices = torch.arange(len(full), device=device)[mask]
        original_partial_indices = torch.arange(len(partial),device=device)
        sorted_masked, associated_partial_rows = lexsort(full[mask])
        assert sorted_masked.equal(partial)
        associated_partial_rows = original_indices[associated_partial_rows].reshape(-1,1)
        new_indices = torch.concatenate((partial, associated_partial_rows), dim=-1).to(torch.uint32).cpu().numpy()
        new_indices[..., -1] -= indexes_ith_chunk_offsets[inverted_fname_index[partial[:,0].cpu()]]
        # new_indices.tofile(f"{INDEXES_LOCATION}/{token_id}_{i}_index_new")
    

100%|█████████████████████████████████████████████| 1/1 [00:01<00:00,  1.69s/it]


In [487]:
new_indices[...,-1]

array([       5,       21,       34, ..., 23524740, 23524742, 23524751],
      dtype=uint32)

In [150]:
INDEXES_LOCATION = 'indexfiles'
y = !find f"{INDEXES_LOCATION}" -name f"{token_id}_*_index"

In [540]:
import subprocess
class DenseEmbeddingsDataset(torch.utils.data.Dataset):
    def __init__(self, token_id, capacity, device, emb_size=2048, nfiles=100):
        global w
        self.token_id = token_id
        self.embedding_size = emb_size
        self.device = device
        self.capacity = capacity
        filenames = list(map(lambda x : x.decode('utf-8').strip(), subprocess.run(["find", INDEXES_LOCATION, "-name", f"{token_id}_*_index_new"], stdout=subprocess.PIPE).stdout.split()))[:nfiles]
        self.informations = self.get_indices(filenames)
        
        # ---------------- NEW GET ITEM BEGIN ----------------
        self.embeddings = np.zeros((len(self), 2048), dtype=np.float16)
        # self.teacher_logits = np.zeros((len(self), 100), dtype=np.float16)
        self.teacher_probs = np.zeros(len(self), dtype=np.float16)
        self.student_probs = np.zeros(len(self), dtype=np.float16)
        w.to(self.device)
        # ----------------------- END ------------------------
    def __len__(self):
        return self.informations.shape[0]
    def pre_read(self):
        nbytes=np.float16().nbytes
        for i, (file_id, example_id, token_pos, student_position) in tqdm(enumerate(self.informations)):
            offset = student_position
            teacher_offset = example_id * 1024 * 100 + token_pos * 100
            gt_offset      = teacher_offset // 100
            # Rank of token_id in the teacher prediction, if any (array of length 1 or 0)
            tid = ((np.fromfile(os.path.join(LOCATION, f"{file_id:04d}_id_t"), dtype=np.int32, offset=teacher_offset*np.int32().nbytes, count=10)) == self.token_id).nonzero()[0]
            # Ground truth at the position we are looking at
            gt  = (np.fromfile(os.path.join(LOCATION, f"{file_id:04d}_id_gt"), dtype=np.int32, offset=gt_offset*np.int32().nbytes, count=1))[0]
            
            gt_adjustment = 0
            if (gt == self.token_id):
                gt_adjustment += 0.2
            else:
                assert len(tid) == 1
                assert tid[0] < 10
            
            # self.teacher_probs[i] = torch.softmax(torch.tensor(np.fromfile(os.path.join(LOCATION, f"{file_id:04d}_logit_t"),dtype=np.float16, offset=teacher_offset*np.float16().nbytes, count=100)), dim=-1)

            # All top_k teacher and student probs
            teacher_probs = torch.softmax(torch.tensor(np.fromfile(os.path.join(LOCATION, f"{file_id:04d}_logit_t"),dtype=np.float16, offset=teacher_offset*np.float16().nbytes, count=100)), dim=-1)
            try:
                student_probs = np.fromfile(os.path.join(LOCATION, f"{file_id:04d}_student_probs_new"), dtype=np.float16, offset=offset*100*np.float16().nbytes,   count=100)
            except:
                student_probs=0
                print(offset)
            if len(tid) == 1: # token_id is in top_k
                teacher_probs[tid[0]] += gt_adjustment
                teacher_probs = (teacher_probs / teacher_probs.sum())[tid[0]] # renormalize in case top 100 teacher probs dont form a prob distribution anymore
                student_probs = student_probs[tid[0]]
            else: # token_id is just the ground truth, but not predicted as top_k by the teacher
                teacher_probs = gt_adjustment
                student_probs = 0
            self.teacher_probs[i] = teacher_probs
            self.student_probs[i] = student_probs
            
            # Old representation with two bytes per element
            # self.embeddings[i] = np.fromfile(os.path.join(LOCATION, f"{file_id:04d}_{self.token_id}_student_embeddings"), dtype=np.float16, offset=offset*2048*np.float16().nbytes,       count=self.embedding_size)
            
            # Multiply by 1024 = 2048 elements, each being 0.5 bytes = 4 bits
            emb = np.fromfile(os.path.join(LOCATION, f"{file_id:04d}_student_embeddings_new"), dtype=np.uint8, offset=offset*1024, count=self.embedding_size//2).reshape(1,-1)
            self.embeddings[i] = w.rescale(q.load_decode(torch.tensor(emb, device=self.device))).cpu().numpy()
            
    def __getitem__(self, i):
        teacher_prob = torch.softmax(torch.tensor(self.teacher_logits[i]), dim=-1).numpy()
        residual_prob = teacher_prob - self.student_prob[i]
        emb = self.embeddings[i]
        offset = self.offset[i]
        return {"embedding" : emb, "residual" : residual_prob, "teacher_prob" : teacher_prob if hyperparams['weighted'] else None, "offset" : offset}
    
    def get_indices(self,filenames):
        holder = [None for _ in range(len(filenames))]
        print(len(filenames))
        for file in filenames:
            index = np.fromfile(file, dtype=np.int16).reshape(-1, 4)
            holder[i] = index
        return np.concatenate(holder)[:self.capacity]


In [541]:
dataset = DenseEmbeddingsDataset(235265, 10**7, 'cuda:1')
dataset.informations = new_indices
dataset.pre_read()

1


7501034it [1:26:05, 1452.04it/s]


In [ ]:
2184343it [24:56, 1344.22it/s]

In [524]:
out = np.unique(stind, axis=0,return_counts=True)

In [528]:
(out[1] > 1).any()

False

In [539]:
new_indices[145717]

array([ 1,  0, 12, 12], dtype=uint32)

In [506]:
helper = (dataset.student_probs == 0).nonzero()[0]
(helper[1:] - helper[:-1] == 1).nonzero()

(array([      5,      64,      76, ..., 7356403, 7356404, 7356405]),)

In [512]:
dataset.student_probs[145717]

0.0

In [513]:
dataset.informations[145717]

array([      1,       0,      12, 1023097], dtype=uint32)

In [504]:
helper

array([    938,    1092,    1133, ..., 7501031, 7501032, 7501033])

In [494]:
new_indices.shape

(7501034, 4)

In [514]:
np.fromfile(os.path.join(LOCATION, f"{file_id:04d}_student_probs_new"), dtype=np.float16, offset=1023097*100*np.float16().nbytes,   count=100)


array([], dtype=float16)

In [516]:
stind[1023097]

IndexError: index 1023097 is out of bounds for axis 0 with size 1023085

In [464]:
stind = np.fromfile(os.path.join(LOCATION, f"{0:04d}_student_index_new"), dtype=np.uint16).reshape(-1,2)

In [476]:
tid[stind[1211,0], stind[1211,1]] == toi

array([False, False, False, False, False, False, False, False,  True,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False])

In [441]:
toi = tokens_of_interest[0]

In [442]:
example_id = 0
token_pos=5
teacher_offset = example_id * 1024 * 100 + token_pos * 100
tid_read = np.fromfile(os.path.join(LOCATION, f"{file_id:04d}_id_t"), dtype=np.int32, offset=teacher_offset*np.int32().nbytes, count=10)


In [443]:
tid = np.fromfile(os.path.join(LOCATION, f"{0:04d}_id_t"), dtype=np.int32).reshape(-1,1024,100)

In [428]:
(tid_read == toi).nonzero()[0]

array([2])

In [439]:
((np.fromfile(os.path.join(LOCATION, f"{file_id:04d}_id_t"), dtype=np.int32, offset=teacher_offset*np.int32().nbytes, count=10)) == toi)

array([False, False,  True, False, False, False, False, False, False,
       False])

In [419]:
tid[0,5]

array([   108,    109, 235265,    708, 235269, 235292, 236338,    603,
          591, 235248,    110,    576,   1049,    791,    192, 235270,
          575,    714,    798,    728,    604, 235316,    139, 124047,
          578,    611,    729,    196,   1457,    892,   3658, 235307,
       235322,    111,    194,    586,    731,    919,    730,   1249,
       235286, 235289, 235371,    727,    878,   1157, 235705,   1420,
       235290, 235403,    195,    590,   1214,   1554,   1707,    774,
          877,   2722,   2804,    651,   3392, 235303,    675,    886,
         1417,   1500,  10200, 235278, 235341,    193,   1165,   2062,
         2439,   2734,   3255,   6572, 235309, 235436,    226,    612,
          955,   1105,   1315,   8331, 178155, 235282, 235298, 235349,
       235421,    577,    689,    749,   1501,   1931,   2456,   2765,
         2889,   6246,  13687,  13992], dtype=int32)

In [431]:
dataset.informations

array([[    0,     0,     5,     5],
       [    0,     0,    21,    21],
       [    0,     0,    34,    34],
       ...,
       [  377,   999,  1010, -2684],
       [  377,   999,  1012, -2682],
       [  377,   999,  1021, -2673]], dtype=int16)

In [39]:
[x.strip() for x in open('file_number_order','r').readlines()] == tt

True

In [393]:
with open('file_number_order', 'w') as f:
    for fnumber in tt:
        f.write(fnumber)
        f.write('\n')

In [90]:
file_0_index = index_token[index_token[:,0]==0]

In [35]:
LOCATION

'/igli/raw_logits_v2'

In [96]:
file_id = 0
token_id = 2167
tid   = torch.tensor(np.fromfile(f"{LOCATION}/{file_id:04d}_id_t", dtype=np.int32).reshape(-1, 1024, 100), device=device)
gtid  = torch.tensor(np.fromfile(f"{LOCATION}/{file_id:04d}_id_gt", dtype=np.int32).reshape(1000, 1024), device=device)
ind   = torch.tensor(np.fromfile(f"{LOCATION}/{file_id:04d}_student_index_new", dtype=np.int16).reshape(-1, 2), device=device)

In [97]:
ind = ind.int()
r_all = (torch.logical_or((tid[ind[:,0], ind[:,1], :10] == token_id).any(axis=-1), (gtid[ind[:,0], ind[:,1]] == token_id))).nonzero().cpu().numpy()   # rows of ind that correspond to the token of interest

In [88]:
index_token = np.fromfile(f"indexfiles/{2167}_0_index", dtype=np.uint16).reshape(-1, 3)

In [89]:
index_token.shape

(351170, 3)

In [105]:
file_0_index

array([[  0,   0, 278],
       [  0,   0, 292],
       [  0,   0, 315],
       ...,
       [  0, 999, 980],
       [  0, 999, 984],
       [  0, 999, 995]], dtype=uint16)

In [112]:
np.unique(file_0_index, axis=0).shape

(6540, 3)

In [109]:
ind[r_all.reshape(-1)][:,0][1:]

tensor([  0,   0,   0,  ..., 999, 999, 999], device='cuda:0',
       dtype=torch.int32)

In [110]:
torch.all(ind[r_all.reshape(-1)][:,0][:-1] <= ind[r_all.reshape(-1)][:,0][1:])

tensor(True, device='cuda:0')

In [95]:
token_id

2167

In [91]:
r_all.shape, file_0_index.shape

((6540, 1), (6982, 3))

In [53]:
tokens = np.zeros((1000,1024,11,1), dtype=np.int32)
ts=np.fromfile(f"{LOCATION}/0000_id_t" , dtype=np.int32).reshape(-1, 1024, 100)[..., :10]
gs=np.fromfile(f"{LOCATION}/0000_id_gt", dtype=np.int32).reshape(1000, 1024)
tokens[:,:,0:10,0]=ts
tokens[:,:,-  1,0]=gs
file_id   =np.tile(np.array([0],dtype=np.uint16).reshape( (1   ,   1,1,1) ), (1000,1024,11,  1) )
example_id=np.tile(np.arange(1000,                     dtype=np.uint16).reshape( (1000,   1,1, 1) ), (   1,1024,11, 1) )
position  =np.tile(np.arange(1024,                     dtype=np.uint16).reshape( (1   ,1024,1, 1) ), (1000,   1,11, 1) )

In [61]:
tokens_cuda = torch.tensor(tokens).to('cuda:3').reshape(-1)
toi_cuda = torch.tensor(tokens_of_interest).to('cuda:3')

In [62]:
IX=torch.argsort(tokens_cuda).cpu().numpy()

In [64]:
file_id    = file_id   .reshape(-1)[IX]
example_id = example_id.reshape(-1)[IX]
position   = position  .reshape(-1)[IX]
tokens     = tokens    .reshape(-1)[IX]

In [65]:
mask = np.isin(tokens, tokens_of_interest)
file_id    = file_id[mask].reshape(-1,1)
example_id = example_id[mask].reshape(-1,1)
position   = position[mask].reshape(-1,1)
indices = np.concatenate((file_id, example_id, position), axis=-1)
tokens     = tokens[mask]

In [69]:
cutoffs = np.where(tokens[:-1] != tokens[1:])[0] + 1


In [71]:
for i,cut in enumerate(cutoffs):
    if tokens[cut] == 2167:
        print(i)
        break

1193


In [82]:
start, end = cutoffs[i], cutoffs[i+1]

In [84]:
indices[start:end].shape

(6982, 3)

In [86]:
(indices[:, 0] == 0).all()

True

In [87]:
indices.dtype

dtype('uint16')

In [41]:
token_id = 2167
dataset = MapReduceEmbeddingDataset(token_id=token_id, device=device, nfiles=10)

10227781it [10:53, 15658.24it/s]


In [112]:
#Multi
class multi_embedding(torch.nn.Module):
    def __init__(self, token_id, device, embedding_size = 2048, number_of_clusters = 2):
        super(multi_embedding, self).__init__()
        self.number_of_clusters=number_of_clusters
        self.embedding_size = embedding_size
        self.normalization_factor = torch.sqrt(torch.tensor(self.embedding_size).to(device))
        self.token_of_interest = token_id
        self.linear1 =torch.nn.Linear(self.embedding_size, self.number_of_clusters, device=device,dtype=torch.float32,bias=True)
        # self.linear2 =torch.nn.Linear(self.number_of_clusters, 1, device=device,dtype=torch.float32,bias=False)
        # with torch.no_grad():
        #     self.linear1.weight.data=(self.linear1.weight*0.01)

        self.device=device
        self.relu=torch.nn.ReLU()
    
    def forward(self, input_embs, student_prob, teacher_prob, weight, percentage=100):
        input_embs = input_embs.to(torch.float32)
        student_prob = student_prob.to(torch.float32)
        teacher_prob = teacher_prob.to(torch.float32)
        print(input_embs, input_embs.shape)
        scores = self.linear1(input_embs)
        scores/=self.normalization_factor
        ps=self.relu(scores)
        print(ps)
        split = int(ps.shape[1] * percentage / 100)
        y = ps[:, :split].sum(dim=-1) - ps[:, split:].sum(dim=-1)
        # y = self.linear2(ps).squeeze()
        # y=torch.maximum(y,-student_prob) # This is not needed anymore i think(?)
        print(y)
        y+=student_prob
        y=torch.clip(y,0,configuration['residual_scaler'])
        print(y)
        # L1
        l1_loss = torch.abs ((teacher_prob - y))
        l1_loss = l1_loss * weight
        # L2
        l2_loss = (teacher_prob - y)**2
        l2_loss = l2_loss * weight
        # KL
        kl_loss = torch.abs(teacher_prob * ((y+1e-9).log() - (teacher_prob+1e-9).log()))
        #default all-zero-l1
        l1_default_zero = torch.abs((teacher_prob - student_prob)) * weight
        #default zero l2
        l2_default_zero = ((teacher_prob - student_prob) ** 2) * weight
        losses = {
            'l1' : l1_loss,
            'l2' : l2_loss,
            'kl' : kl_loss,
            'l1_0':l1_default_zero,
            'l2_0':l2_default_zero,
        }
        
        return y, losses

class multi_embedding_withbias(torch.nn.Module):
    def __init__(self, token_id, device, embedding_size = 2048, number_of_clusters = 2):
        super(multi_embedding_withbias, self).__init__()
        self.number_of_clusters=number_of_clusters
        self.embedding_size = embedding_size
        self.normalization_factor = torch.sqrt(torch.tensor(self.embedding_size).to(device))
        self.token_of_interest = token_id
        self.linear1 =torch.nn.Linear(self.embedding_size, self.number_of_clusters, device=device,dtype=torch.float32,bias=True)
        # self.linear2 =torch.nn.Linear(self.number_of_clusters, 1, device=device,dtype=torch.float32,bias=False)
        # with torch.no_grad():
        #     self.linear1.weight.data=(self.linear1.weight*0.01)

        self.device=device
        self.relu=torch.nn.ReLU()
    
    def forward(self, input_embs, res_prob, teacher_prob):
        input_embs = input_embs.to(torch.float32)
        res_prob = res_prob.to(torch.float32)
        teacher_prob = teacher_prob.to(torch.float32)
        # print('input_embs',input_embs.shape)
        scores = self.linear1(input_embs)
        scores/=self.normalization_factor
        #print('scores',scores.shape)
        # ps=torch.softmax(scores,dim=-1)
        ps=self.relu(scores)
        # print(ps)
        #print('ps',ps.shape)
        # y =self.linear2(ps)/50
        # y =(ps[:,0:self.number_of_clusters//2].sum(dim=-1,keepdims=True)-ps[:,self.number_of_clusters//2:self.number_of_clusters].sum(dim=-1,keepdims=True))
        y = ps
        # print('y',y.shape)
        # loss = torch.abs ((res_prob- y))
        loss = ((res_prob- y)**2)
        loss = loss * teacher_prob
        return y, loss


In [39]:
for cut0, cut1, cut2, cut3 in tests:
    # mix = torch.zeros_like(allemb_)
    # mix[:,:cut0] = allemb_[:,:cut0]
    # mix[:,cut0:cut0+cut1] = quant4[:,cut0:cut0+cut1]
    # mix[:,cut0+cut1:cut0+cut1+cut2] = quant2[:,cut0+cut1:cut0+cut1+cut2]
    # mix[:,cut0+cut1+cut2:cut0+cut1+cut2+cut3] = quant1[:,cut0+cut1+cut2:cut0+cut1+cut2+cut3]
    # data = (allemb_, mix[:660000], tp__, sp__)
    data = (allemb_, quant4torch[:660000], tp__, sp__)
    torch.cuda.empty_cache()
    configuration['percentage'] = 100
    configuration['epochs'] = 50
    configuration['target'] = 'residual'
    configuration['residual_scaler'] = 1
    configuration['num_clusters'] = [16]
    configuration['lr'] = 0.0002
    configuration['loss'] = 'l1'
    configuration['weight_decay'] = 1
    configuration['note'] = f'{configuration["percentage"]}_percent_residual={configuration["residual_scaler"]}'
    trainer.train(configuration, data=data)
    torch.cuda.empty_cache()

132it [00:11, 11.82it/s]
132it [00:05, 23.63it/s]
132it [00:05, 23.36it/s]
132it [00:10, 12.25it/s]
132it [00:05, 23.59it/s]
132it [00:05, 23.43it/s]
132it [00:11, 11.97it/s]
132it [00:05, 23.66it/s]
132it [00:05, 23.44it/s]
132it [00:11, 11.84it/s]
132it [00:05, 23.78it/s]
132it [00:05, 23.48it/s]
132it [00:11, 11.84it/s]
132it [00:05, 23.91it/s]
132it [00:05, 23.75it/s]
132it [00:11, 11.94it/s]
132it [00:05, 23.32it/s]
132it [00:05, 24.30it/s]
132it [00:11, 11.99it/s]
132it [00:05, 23.83it/s]
132it [00:05, 23.63it/s]
132it [00:11, 11.94it/s]
132it [00:05, 23.47it/s]
132it [00:05, 24.12it/s]
132it [00:05, 23.31it/s]
132it [00:11, 11.95it/s]
132it [00:05, 23.83it/s]
132it [00:05, 23.41it/s]
132it [00:11, 11.97it/s]
132it [00:05, 23.19it/s]
132it [00:05, 24.12it/s]
132it [00:11, 11.74it/s]
132it [00:05, 24.09it/s]
132it [00:05, 23.73it/s]
132it [00:11, 11.82it/s]
132it [00:05, 23.34it/s]
132it [00:05, 23.42it/s]
132it [00:10, 12.04it/s]
132it [00:05, 23.27it/s]
132it [00:05, 24.30it/s]
